In [ ]:
library(tidyverse)

# ============================================================
# 1. Paths
# ============================================================
H_ROOT <- paste0(
  "/project/zduren/durenlab/naqing_palmetto/REGA_separate/",
  "03_process_rega/matrices"
)

METADATA_ROOT <- paste0(
  "/project/zduren/durenlab/naqing_palmetto/REGA_separate/",
  "01_prepare/data/metadata_by_group"
)

SELECTED_FILE <- "/project/zduren/durenlab/naqing_palmetto/REGA_separate/02_run_rega/pathology_association/selected_module.txt"

OUT_DIR <- dirname(SELECTED_FILE)


In [ ]:
# ============================================================
# 2. Read selected module names
# ============================================================
# Expected examples:
# Ast_AG_M6
# Mic_Immune_EC_M13
selected_names <- readLines(SELECTED_FILE) %>%
  trimws() %>%
  .[. != ""]

if (length(selected_names) == 0) {
  stop("No modules were found in /project/zduren/durenlab/naqing_palmetto/REGA_separate/02_run_rega/pathology_association/figures/selected_modules.txt")
}


In [ ]:
selected_names

In [ ]:
# ============================================================
# 3. Function to read one selected module
# ============================================================
read_one_module <- function(selected_name) {

  # Extract module from the end of the name, for example M6.
  module <- str_extract(selected_name, "M[0-9]+$")

  # Remove _M6, then extract the region from the end.
  celltype_region <- str_remove(selected_name, "_M[0-9]+$")
  region <- str_extract(celltype_region, "[^_]+$")

  # Everything before the final region is the cell type.
  # This also works for Mic_Immune.
  celltype <- str_remove(celltype_region, paste0("_", region, "$"))

  H_file <- file.path(
    H_ROOT,
    celltype_region,
    paste0(celltype_region, "_H_module_by_sample.tsv")
  )

  metadata_file <- file.path(
    METADATA_ROOT,
    paste0(celltype_region, "_metadata.tsv")
  )

  if (!file.exists(H_file)) {
    warning("H file not found: ", H_file)
    return(NULL)
  }

  if (!file.exists(metadata_file)) {
    warning("Metadata file not found: ", metadata_file)
    return(NULL)
  }

  # H is expected to have modules in rows and individuals in columns.
  H <- read.delim(
    H_file,
    header = TRUE,
    sep = "\t",
    check.names = FALSE,
    stringsAsFactors = FALSE
  )

  # The first column contains module names.
  colnames(H)[1] <- "Module"

  if (!module %in% H$Module) {
    warning(module, " was not found in: ", H_file)
    return(NULL)
  }

  # Select one module and convert sample columns to long format.
  H_long <- H %>%
    filter(Module == module) %>%
    pivot_longer(
      cols = -Module,
      names_to = "ROSMAP_IndividualID",
      values_to = "module_activity"
    ) %>%
    mutate(module_activity = as.numeric(module_activity))

  metadata <- read.delim(
    metadata_file,
    header = TRUE,
    sep = "\t",
    check.names = FALSE,
    stringsAsFactors = FALSE
  )

  # Match H samples to metadata individuals.
  result <- H_long %>%
    inner_join(
      metadata %>% select(ROSMAP_IndividualID, Pathology),
      by = "ROSMAP_IndividualID"
    ) %>%
    filter(Pathology %in% c("nonAD", "earlyAD", "lateAD")) %>%
    mutate(
      selected_module = selected_name,
      celltype = celltype,
      region = region,
      Pathology = factor(
        Pathology,
        levels = c("nonAD", "earlyAD", "lateAD")
      )
    )

  return(result)
}


In [ ]:
# ============================================================
# 4. Read all selected modules
# ============================================================
plot_data <- map_dfr(selected_names, read_one_module)

if (nrow(plot_data) == 0) {
  stop("No H values could be matched to the metadata.")
}

# Keep x-axis modules in the same order as /project/zduren/durenlab/naqing_palmetto/REGA_separate/02_run_rega/pathology_association/figures/selected_modules.txt.
plot_data$selected_module <- factor(
  plot_data$selected_module,
  levels = selected_names
)

# # Save the plotting table for checking.
# write.table(
#   plot_data,
#   file.path(OUT_DIR, "selected_module_activity_plot_data.tsv"),
#   sep = "\t",
#   quote = FALSE,
#   row.names = FALSE
# )

In [ ]:
# ============================================================
# 5. Plot boxplots
# ============================================================
# Colorblind-friendly, softer colors.
pathology_colors <- c(
  "nonAD" = "#66AABB",
  "earlyAD" = "#EEAA55",
  "lateAD" = "#CC6677"
)


In [ ]:
# Calculate the visible range from the boxplot whiskers.
# This prevents hidden extreme outliers from creating a large white area.
whisker_limits <- plot_data %>%
  group_by(selected_module, Pathology) %>%
  summarise(
    whisker_low = boxplot.stats(module_activity)$stats[1],
    whisker_high = boxplot.stats(module_activity)$stats[5],
    .groups = "drop"
  )

y_min <- min(whisker_limits$whisker_low, na.rm = TRUE)
y_max <- max(whisker_limits$whisker_high, na.rm = TRUE)
y_padding <- (y_max - y_min) * 0.06

# Avoid zero padding if all values happen to be identical.
if (y_padding == 0) {
  y_padding <- 0.1
}

p <- ggplot(
  plot_data,
  aes(x = selected_module, y = module_activity, fill = Pathology)
) +
  geom_boxplot(
    position = position_dodge(width = 0.80),
    width = 0.68,
    outlier.shape = NA,
    linewidth = 0.55
  ) +
  scale_fill_manual(values = pathology_colors, drop = FALSE) +
  scale_y_continuous(expand = expansion(mult = c(0, 0))) +
  coord_cartesian(
    ylim = c(y_min - y_padding, y_max + y_padding),
    clip = "on"
  ) +
  labs(
    x = NULL,
    y = "Module activity",
    fill = "Pathology"
  ) +
  theme_classic(base_size = 18) +
  theme(
    axis.text.x = element_text(
      angle = 90,
      hjust = 1,
      vjust = 0.5,
      color = "black",
      size = 15
    ),
    axis.text.y = element_text(color = "black", size = 16),
    axis.title.y = element_text(face = "bold", size = 20),
    legend.text = element_text(size = 16),
    legend.title = element_text(face = "bold", size = 18),
    legend.position = "top",
    plot.margin = margin(6, 12, 6, 8)
  )

In [ ]:
p

In [ ]:
# Make the width increase when more modules are selected.
plot_width <- max(10, length(selected_names) * 0.65)


In [ ]:
ggsave(
  file.path(OUT_DIR, "selected_module_activity_boxplots_add_1neg.pdf"),
  p,
  width = plot_width,
  height = 6.5,
  limitsize = FALSE
)